# Punjab Traffic Accident Severity Prediction

End-to-end workflow: load the accident-level dataset, explore it, engineer/select features, train and compare several classification models (with full `classification_report`s), and visualize accident hotspots on a map.

**Sections**
1. Setup & Imports
2. Data Loading
3. Exploratory Data Analysis
4. Data Preprocessing
5. Feature Selection
6. Train-Test Split
7. Model Training & Evaluation
8. Geographic Hotspot Visualization
9. Saving the Best Model

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap, MarkerCluster

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

sns.set_style('whitegrid')
RANDOM_STATE = 42

## 2. Data Loading

In [ ]:
df = pd.read_csv('dataset.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

### 2.1 Data Quality Checks

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

## 3. Exploratory Data Analysis

### 3.1 Target Variable: Severity Distribution

In [ ]:
df['Severity'].value_counts().plot(kind='bar')
plt.title('Severity Distribution')
plt.xlabel('Severity')
plt.ylabel('Count')
plt.show()

### 3.2 Weather Distribution

In [ ]:
plt.figure(figsize=(8,8))
df['Weather'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Weather Distribution')
plt.xlabel('')
plt.ylabel('')
plt.show()

### 3.3 Traffic Violation vs Severity

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(x='Traffic_Violation', hue='Severity', data=df)
plt.title('Traffic Violation vs Severity')
plt.xlabel('Traffic Violation')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.4 Speed Limit vs Severity

In [ ]:
sns.histplot(data=df, x='Speed_Limit', hue='Severity', multiple='stack', bins=10)
plt.title('Speed Limit vs Severity')
plt.xlabel('Speed Limit')
plt.ylabel('Count')
plt.show()

### 3.5 Traffic Density vs Severity

In [ ]:
plt.figure(figsize=(8,5))
sns.heatmap(pd.crosstab(df['Traffic_Density'], df['Severity']), annot=True, fmt='d', cmap='YlGnBu')
plt.title('Traffic Density vs Severity')
plt.xlabel('Severity')
plt.ylabel('Traffic Density')
plt.show()

### 3.6 Vehicle Type vs Severity

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(x='Vehicle_Type', hue='Severity', data=df)
plt.title('Vehicle Type vs Severity')
plt.xlabel('Vehicle Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.7 Road Condition vs Severity

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(x='Road_Condition', hue='Severity', data=df, multiple='stack')
plt.title('Road Condition vs Severity')
plt.xlabel('Road Condition')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.8 Accidents by Day of Week

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(x='Day_of_Week', hue='Severity', data=df,
              order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
plt.title('Number of Accidents by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Count')
plt.show()

## 4. Data Preprocessing

In [ ]:
# Converting object variables to category dtype
object_columns = df.select_dtypes(include=['object']).columns
for col in object_columns:
    df[col] = df[col].astype('category')

In [ ]:
# Encoding categorical columns
encoder = LabelEncoder()
categorical_columns = df.select_dtypes(include=['category']).columns
for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])

In [ ]:
df.head()

## 5. Feature Selection

In [ ]:
# Correlation matrix
corr_matrix = df.corr(numeric_only=True)
plt.figure(figsize=(30, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Correlation of each feature with the target 'Severity' (as a percentage), sorted
correlation_with_severity = (corr_matrix['Severity'].sort_values(ascending=False) * 100)
correlation_with_severity

In [ ]:
# Selecting the features for the model
col = [
    'Weather',
    'Road_Condition',
    'Light_Condition',
    'Traffic_Density',
    'Speed_Limit',
    'Road_Type',
    'Vehicle_Type',
    'Traffic_Violation',
    'Accident_Cause',
    'Injuries',
    'Fatalities',
    'Fatal_Status',
    'Total_Casualties',
    'Severity',
]
sel_df = df[col]

## 6. Train-Test Split

In [ ]:
X = sel_df.drop('Severity', axis=1)
y = sel_df['Severity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape

## 7. Model Training & Evaluation

We train several classification models and evaluate each one with accuracy and a full `classification_report` (precision / recall / F1 per class), so they can be compared on equal footing.

### 7.1 Define the models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes': GaussianNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=3,
        random_state=RANDOM_STATE,
        eval_metric='mlogloss',
    ),
}

### 7.2 Train and evaluate each model

Support Vector Machines scale poorly to tens of thousands of rows (training time grows roughly quadratically), so SVC is fit on a stratified subsample for speed; every other model uses the full training set.

In [ ]:
fitted_models = {}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    fitted_models[name] = model
    results.append({'Model': name, 'Accuracy': acc})

    print('=' * 60)
    print(name)
    print('=' * 60)
    print(f'Accuracy: {acc:.4f}\n')
    print(classification_report(y_test, y_pred))
    print()

In [ ]:
# Support Vector Machine - trained on a stratified subsample for tractable runtime
SVM_SAMPLE_SIZE = 15000
if len(X_train) > SVM_SAMPLE_SIZE:
    X_train_svm, _, y_train_svm, _ = train_test_split(
        X_train, y_train,
        train_size=SVM_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )
else:
    X_train_svm, y_train_svm = X_train, y_train

svm_model = SVC(kernel='rbf', random_state=RANDOM_STATE)
svm_model.fit(X_train_svm, y_train_svm)
y_pred_svm = svm_model.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)

fitted_models['Support Vector Machine'] = svm_model
results.append({'Model': 'Support Vector Machine', 'Accuracy': acc_svm})

print('=' * 60)
print('Support Vector Machine (trained on subsample)')
print('=' * 60)
print(f'Accuracy: {acc_svm:.4f}\n')
print(classification_report(y_test, y_pred_svm))

### 7.3 Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False).reset_index(drop=True)
results_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x='Accuracy', y='Model', palette='viridis')
plt.title('Model Comparison - Test Accuracy')
plt.xlabel('Accuracy')
plt.ylabel('')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

### 7.4 Confusion Matrix - Best Model

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = fitted_models[best_name]
y_pred_best = best_model.predict(X_test)

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best, cmap='Blues')
plt.title(f'Confusion Matrix - {best_name}')
plt.show()

print(f'Best model: {best_name} (Accuracy: {results_df.iloc[0]["Accuracy"]:.4f})')

## 8. Geographic Hotspot Visualization

### 8.1 Accidents by Severity (sample)

Plotting every accident as an individual, unclustered marker doesn't scale to tens of thousands of points, so this map uses a random sample for a responsive view. See 8.3/8.4 below for full-data views (heatmap and clustering).

In [ ]:
SAMPLE_SIZE = 5000
map_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)

m = folium.Map(location=[df['Latitude'].mean(), df['Longitude'].mean()], zoom_start=6)
for _, row in map_df.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5,
        color='red' if row['Severity'] == 1 else 'orange' if row['Severity'] == 2 else 'green',
        fill=True,
        fill_color='red' if row['Severity'] == 1 else 'orange' if row['Severity'] == 2 else 'green',
    ).add_to(m)

### 8.2 Hotspot Locations

In [ ]:
hotspots = df[df['Accident_Hotspot'] == 1]

m = folium.Map(
    location=[hotspots['Latitude'].mean(), hotspots['Longitude'].mean()],
    zoom_start=7,
)

for _, row in hotspots.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Severity: {row['Severity']}",
        icon=folium.Icon(color='red', icon='warning-sign'),
    ).add_to(m)

m

### 8.3 Heatmap (all accidents)

In [ ]:
m = folium.Map(location=[df['Latitude'].mean(), df['Longitude'].mean()], zoom_start=6)
HeatMap(df[['Latitude', 'Longitude']].values.tolist(), radius=10).add_to(m)
m

### 8.4 Marker Clusters (all accidents)

In [ ]:
m = folium.Map(location=[df['Latitude'].mean(), df['Longitude'].mean()], zoom_start=6)
cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    folium.Marker(
        [row['Latitude'], row['Longitude']],
        popup=str(row['Severity']),
    ).add_to(cluster)

m

## 9. Saving the Best Model